# DEBUG_stage2_DiagnoseFgcmVisitCatalog: which stage-1 visit summary makes fgcmBuildFromIsolatedStar fail?

The quantum fails in `fgcmcal/fgcmBuildStarsBase.py`, `_fillVisitCatalog`, at `rec['psfSigma'] = psfSigma.item()` with `ValueError: can only convert an array of size 1 to a Python scalar`.

`_fillVisitCatalog` loops over all the `preliminary_visit_summary` of the stage-1 collection (connection `visit_summaries`), and for each visit computes the median PSF sigma over the detectors:

```
psfSigmas = summary['psfSigma']
goodSigma = (nan_to_num(psfSigmas) > 0) & (nan_to_num(pixelScales) > 0)
if goodSigma.size > 2:   psfSigma = np.median(psfSigmas[goodSigma])
elif goodSigma.size > 0: psfSigma = psfSigmas[goodSigma][0]
else:                    psfSigma = 0.0
rec['psfSigma'] = psfSigma.item()
```

For a plain one-dimensional float column `psfSigma` is always a scalar, so one visit summary must have an unexpected `psfSigma` column (shape, dtype), or a case this code does not handle. This notebook replays exactly this computation on **every** stage-1 visit summary, catches the exception per visit, and reports the offending visits and what is special about them.

Notes on the code above: `goodSigma.size` is the number of detector rows of the summary, not the number of good detectors, so the `else` branch (empty summary) and the `goodSigma.size > 2` branch with zero good detectors are not what they look like; the replay records them too.

Run with the `lsst_distrib` kernel at CC-IN2P3 (butler `main`).

- author : Sylvie Dagoret-Campagne
- creation date : 2026-11-17
- last update : 2026-11-17
- input collection : `u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage1`
- failing log : `/sps/lsst/users/dagoret/bps/submit/u/dagoret/LSSTCam/runcosmos01_20261103_tract9813patchsel_RunCosmos9patches/stage2/20261105T173215Z/logs/fgcmBuildFromIsolatedStar/`
- see also : `04_ScanVisitSummaries.ipynb`, `05_CheckStage1AllOutputs.ipynb`

In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

SRC = Path('../../../../src').resolve()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from rubincc import monster as mon

NB_TAG = 'DEBUG_stage2_DiagnoseFgcmVisitCatalog'
DATA_DIR = Path(f'data_{NB_TAG}')
DATA_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)

REPO = 'main'
USER = os.getenv('USER')
INSTRUMENT = 'LSSTCam'
RUN_TAG = 'runcosmos01_20261116_tract9813_RunCosmos9patches'
STAGE1 = f'u/{USER}/LSSTCam/{RUN_TAG}/stage1'
DEFAULTS = 'LSSTCam/defaults'

DEFAULT_PIXEL_SCALE = 0.2   # arcsec/pixel, only used if the summary has no pixelScale column
STRIDE = 1                  # 1 = all visits; use a larger value for a quick test

## 1. The visit summaries read by fgcm

In [2]:
butler = mon.get_butler(REPO, collections=[STAGE1, DEFAULTS])
reg = butler.registry

refs = sorted(reg.queryDatasets('preliminary_visit_summary', collections=[STAGE1], findFirst=True),
              key=lambda r: r.dataId['visit'])
print(len(refs), 'preliminary_visit_summary datasets in', STAGE1)
print('runs:', sorted({ref.run for ref in refs}))
refs = refs[::STRIDE]

2357 preliminary_visit_summary datasets in u/dagoret/LSSTCam/runcosmos01_20261116_tract9813_RunCosmos9patches/stage1
runs: ['u/dagoret/LSSTCam/runcosmos01_20261116_tract9813_RunCosmos9patches/stage1/20261116T154231Z']


## 2. Replay of the psfSigma computation of `_fillVisitCatalog` on every visit

One row per visit: number of detector rows, shape / dtype of the `psfSigma` column, number of good detectors, type and size of the resulting `psfSigma`, and the exception raised by `.item()` if any. The skyBg lines of the same function are replayed too.

In [3]:
def replay_visit(ref):
    rec = dict(visit=int(ref.dataId['visit']), run=ref.run, error='')
    try:
        summary = butler.get(ref)
        rec['n_rows'] = len(summary)
        psf = summary['psfSigma']
        rec['psf_shape'] = str(psf.shape)
        rec['psf_dtype'] = str(psf.dtype)
        rec['has_pixelScale'] = 'pixelScale' in summary.schema
        if rec['has_pixelScale']:
            pixel_scales = summary['pixelScale']
        else:
            pixel_scales = np.full(len(psf), DEFAULT_PIXEL_SCALE)
        sigmas = np.array(psf, copy=True)
        good = (np.nan_to_num(sigmas) > 0) & (np.nan_to_num(pixel_scales) > 0)
        rec['mask_size'] = int(good.size)
        rec['n_good_sigma'] = int(good.sum())
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            if good.size > 2:
                value = np.median(sigmas[good])
            elif good.size > 0:
                value = sigmas[good][0]
            else:
                value = 0.0
        rec['value_type'] = type(value).__name__
        rec['value_size'] = int(np.size(value))
        rec['psfSigma'] = float(value) if np.size(value) == 1 else np.nan
        value.item()   # the call that fails in fgcmBuildStarsBase.py line 354
        # skyBg lines of the same function
        sky = summary['skyBg']
        rec['n_good_skyBg'] = int((np.nan_to_num(sky) > 0.0).sum())
        rec['max_detector_id'] = int(np.max(summary['id']))
        rec['band'] = str(summary[0]['band']) if 'band' in summary.schema else ''
    except Exception as exc:
        rec['error'] = f'{type(exc).__name__}: {str(exc)[:150]}'
    return rec


replay = pd.DataFrame([replay_visit(ref) for ref in refs])
print(len(replay), 'visits replayed;', int((replay.error != '').sum()), 'raise an exception')
replay.to_csv(DATA_DIR / 'replay_psfSigma.csv', index=False)

2357 visits replayed; 1 raise an exception


## 3. Offending visits

In [4]:
print('exceptions:')
print(replay.error.value_counts().to_string())
print()
print('psfSigma column shape / dtype:')
print(replay.groupby(['psf_shape', 'psf_dtype']).size().sort_values(ascending=False).head(10).to_string())
print()
print('type and size of the computed psfSigma:')
print(replay.groupby(['value_type', 'value_size']).size().to_string())
print()
print('number of detector rows per visit summary:')
print(replay.n_rows.describe().to_string())

suspects = replay[(replay.error != '') | (replay.value_size != 1) | (replay.n_good_sigma == 0)]
print()
print(len(suspects), 'suspect visits (exception, non-scalar psfSigma, or no good psfSigma):')
display(suspects)

exceptions:
error
                                                                                                                                                                      2356
ValueError: Failure from formatter 'lsst.obs.base.formatters.fitsGeneric.FitsGenericFormatter' for dataset 01a1e517-59a3-78b1-94ad-34766c813f64 (preliminary_visit       1

psfSigma column shape / dtype:
psf_shape  psf_dtype
(73,)      float32      442
(72,)      float32      437
(71,)      float32      383
(69,)      float32      298
(70,)      float32      279
(68,)      float32      190
(67,)      float32       79
(74,)      float32       57
(64,)      float32       56
(75,)      float32       20

type and size of the computed psfSigma:
value_type  value_size
float32     1.0           2356

number of detector rows per visit summary:
count    2356.000000
mean       69.676995
std         7.685683
min         3.000000
25%        69.000000
50%        71.000000
75%        72.000000
max        84.000000


,visit,run,error,n_rows,psf_shape,psf_dtype,has_pixelScale,mask_size,n_good_sigma,value_type,value_size,psfSigma,n_good_skyBg,max_detector_id,band
160,2025052000143,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,ValueError: Failure from formatter 'lsst.obs.b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
252,2025052300106,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,,76.0,"(76,)",float32,True,76.0,0.0,float32,1.0,NaN,76.0,180.0,u
253,2025052300107,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,,71.0,"(71,)",float32,True,71.0,0.0,float32,1.0,NaN,71.0,173.0,u
254,2025052300108,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,,73.0,"(73,)",float32,True,73.0,0.0,float32,1.0,NaN,73.0,172.0,u
255,2025052300109,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,,73.0,"(73,)",float32,True,73.0,0.0,float32,1.0,NaN,73.0,175.0,u
2100,2026030900025,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,,73.0,"(73,)",float32,True,73.0,0.0,float32,1.0,NaN,73.0,180.0,i
2155,2026041000266,u/dagoret/LSSTCam/runcosmos01_20261116_tract98...,,73.0,"(73,)",float32,True,73.0,0.0,float32,1.0,NaN,73.0,171.0,g


## 4. Look inside the first suspect summary

Schema names, the raw `psfSigma` and `pixelScale` columns, and the detectors present.

In [5]:
if len(suspects):
    visit = int(suspects.iloc[0]['visit'])
    ref = next(r for r in refs if int(r.dataId['visit']) == visit)
    summary = butler.get(ref)
    print('visit', visit, ' run', ref.run, ' rows', len(summary))
    print('schema names:', list(summary.schema.getNames()))
    print('detector ids:', list(np.asarray(summary['id'])))
    print('psfSigma raw:', np.asarray(summary['psfSigma']))
    if 'pixelScale' in summary.schema:
        print('pixelScale raw:', np.asarray(summary['pixelScale']))
    print('skyBg raw:', np.asarray(summary['skyBg']))
else:
    print('no suspect visit: every summary replays without exception. The failure is then not in the content of a single summary;')
    print('compare the installed fgcmcal source (see traceback line 354) with the code quoted in the first cell.')

visit 2025052000143  run u/dagoret/LSSTCam/runcosmos01_20261116_tract9813_RunCosmos9patches/stage1/20261116T154231Z  rows 71
schema names: ['starComa2Median', 'starComa1Median', 'psfTE3ex', 'magLim', 'psfArea', 'starEMedian', 'starE41Median', 'preliminary_wcs_visit_pointing_residual', 'starTrefoil2Median', 'skyBg', 'psfTE2e1', 'psfAdaptiveThresholdValue', 'raCorners', 'centroidDiffShapeletsVsSlotMedian', 'shapeletsOnlyIqScore', 'dec', 'psfTE4ex', 'physical_filter', 'psfIxx', 'psfTE1ex', 'band', 'expTime', 'shapeletsStarUnNormalizedEMedian', 'bbox_min_x', 'effTimeSkyBgScale', 'visit', 'nPsfStar', 'shapeletsCoeffs', 'ra', 'psfIyy', 'psfTE1e2', 'preliminary_wcs_detector_pointing_rejected', 'psfStarDeltaE2Median', 'zeroPoint', 'shapeletsStarEMedian', 'psfTE3e1', 'nShapeletsStar', 'psfStarDeltaSizeMedian', 'bbox_max_y', 'decCorners', 'pixelScale', 'astromOffsetStd', 'astromOffsetMean', 'psfApFluxDelta', 'psfIxy', 'maxDistToNearestPsf', 'psfAdaptiveIncludeThresholdMultiplier', 'refCatSourceD

## 5. Are the suspects related to the selection (bands, nights, number of detectors)?

Compares the suspect visits with the others: band, mid-exposure MJD and number of detector rows. Useful to decide whether to exclude them from the stage-2 data query or to rerun the stage-1 tasks for these visits.

In [6]:
visit_ids = [int(v) for v in replay['visit']]
records = reg.queryDimensionRecords('exposure', where='instrument = inst AND exposure IN (ids)',
                                    bind={'inst': INSTRUMENT, 'ids': visit_ids})
meta = pd.DataFrame([{'visit': rec.id, 'physical_filter': rec.physical_filter, 'day_obs': rec.day_obs,
                      'exposure_time': getattr(rec, 'exposure_time', np.nan),
                      'observation_type': getattr(rec, 'observation_type', '')} for rec in records])
full = replay.merge(meta, on='visit', how='left')
full['suspect'] = full.visit.isin(suspects.visit)

print('suspects per physical filter:')
print(full.groupby('physical_filter')['suspect'].agg(['sum', 'count']).to_string())
print()
print('suspects per day_obs:')
print(full.groupby('day_obs')['suspect'].agg(n_suspect='sum', n_visits='count').query('n_suspect > 0').to_string())
print()
print('rows per summary, suspects vs others:')
print(full.groupby('suspect')['n_rows'].describe().to_string())
full.to_csv(DATA_DIR / 'replay_with_metadata.csv', index=False)

suspects per physical filter:
                 sum  count
physical_filter            
g_6                1    401
i_39               1    705
r_57               0    488
u_24               5    205
y_10               0    135
z_20               0    423

suspects per day_obs:
          n_suspect  n_visits
day_obs                      
20250520          1       104
20250523          4       118
20260309          1         5
20260410          1        14

rows per summary, suspects vs others:
          count       mean       std   min   25%   50%   75%   max
suspect                                                           
False    2350.0  69.668085  7.693111   3.0  69.0  71.0  72.0  84.0
True        6.0  73.166667  1.602082  71.0  73.0  73.0  73.0  76.0


## 6. The suspect visits in detail, and an exclusion expression for the stage-2 data query

Number of finite values and median of the main summary quantities for each suspect visit (a visit whose PSF could not be measured has `psfSigma` and `pixelScale` all NaN). The last cell writes a `NOT IN` expression to add to the stage-2 data query.

In [7]:
cols_show = ['nPsfStar', 'zeroPoint', 'magLim', 'effTime', 'expTime', 'zenithDistance', 'skyBg', 'psfSigma', 'pixelScale']
rows_s = []
for visit in suspects['visit']:
    ref = next(r for r in refs if int(r.dataId['visit']) == int(visit))
    summary = butler.get(ref)
    row = dict(visit=int(visit), n_det=len(summary), band=str(summary[0]['band']))
    for col in cols_show:
        if col in summary.schema:
            vals = np.asarray(summary[col], dtype=float)
            row[col + '_n_finite'] = int(np.isfinite(vals).sum())
            row[col + '_median'] = np.nanmedian(vals) if np.isfinite(vals).any() else np.nan
    rows_s.append(row)
suspect_details = pd.DataFrame(rows_s)
display(suspect_details)
suspect_details.to_csv(DATA_DIR / 'suspect_visit_details.csv', index=False)

bad_visits = sorted(int(v) for v in suspects['visit'])
exclude_expr = 'exposure NOT IN (' + ', '.join(str(v) for v in bad_visits) + ')'
(DATA_DIR / 'visits_to_exclude.txt').write_text(exclude_expr + chr(10))
print(exclude_expr)

,visit,n_det,band,nPsfStar_n_finite,nPsfStar_median,zeroPoint_n_finite,zeroPoint_median,magLim_n_finite,magLim_median,effTime_n_finite,...,expTime_n_finite,expTime_median,zenithDistance_n_finite,zenithDistance_median,skyBg_n_finite,skyBg_median,psfSigma_n_finite,psfSigma_median,pixelScale_n_finite,pixelScale_median
0,2025052000143,71,u,71,35.0,71,30.326677,71,23.32691,71,...,71,38.001133,71,39.870678,71,56.746506,71,3.350210,71,0.200203
1,2025052300106,76,u,76,0.0,0,NaN,0,NaN,0,...,76,38.001022,0,NaN,76,89.203445,0,NaN,0,NaN
2,2025052300107,71,u,71,0.0,0,NaN,0,NaN,0,...,71,38.000065,0,NaN,71,89.458145,0,NaN,0,NaN
3,2025052300108,73,u,73,0.0,0,NaN,0,NaN,0,...,73,38.001015,0,NaN,73,89.700249,0,NaN,0,NaN
4,2025052300109,73,u,73,0.0,0,NaN,0,NaN,0,...,73,38.001080,0,NaN,73,89.268997,0,NaN,0,NaN
5,2026030900025,73,i,73,0.0,0,NaN,0,NaN,0,...,73,30.001060,0,NaN,73,2258.026855,15,4.566674,0,NaN
6,2026041000266,73,g,73,0.0,0,NaN,0,NaN,0,...,73,30.000124,0,NaN,73,668.921753,0,NaN,0,NaN


exposure NOT IN (2025052000143, 2025052300106, 2025052300107, 2025052300108, 2025052300109, 2026030900025, 2026041000266)


## 7. Run the real fgcmcal code on every visit

Section 2 replays the psfSigma logic by hand and finds no exception, so something differs between that replay and the real quantum. Here the **real** `fgcmMakeVisitCatalog` of the installed `fgcmcal` is called, one visit at a time, with the camera and for both values of `useScienceDetectors` (the task picks it by comparing the number of detectors of the camera with `nCcd` of the look-up table). It also prints the numpy version and the fgcmcal location of this kernel: compare them with the software used by the bps job (`/opt/lsst/software/stack/...` in the traceback).

In [ ]:
import logging
import traceback

import lsst.fgcmcal
from lsst.pipe.base import Pipeline

logging.getLogger('lsst').setLevel(logging.WARNING)
print('numpy', np.__version__, '| fgcmcal imported from', lsst.fgcmcal.__file__)

try:
    camera = butler.get('camera', instrument=INSTRUMENT, exposure=int(refs[0].dataId['visit']))
except Exception:
    from lsst.obs.lsst import LsstCam
    camera = LsstCam().getCamera()

lut_nccd = None
try:
    lut = butler.get('fgcmLookUpTable', instrument=INSTRUMENT)
    lut_nccd = int(lut[0]['nCcd'])
    del lut
except Exception as exc:
    print('fgcmLookUpTable not read:', type(exc).__name__, str(exc)[:150])
print('detectors in the camera:', len(camera), '| nCcd in the look-up table:', lut_nccd)

# the task must be configured as in the pipeline: a default config has no primaryBands and fails validation
drp_pipe_dir = os.getenv('DRP_PIPE_DIR')
if drp_pipe_dir and (Path(drp_pipe_dir) / 'pipelines/LSSTCam/DRP.yaml').exists():
    pipeline_file = Path(drp_pipe_dir) / 'pipelines/LSSTCam/DRP.yaml'
else:
    pipeline_file = Path('../fig_diagrams/pipeline_drp_pipe_LSSTCam_DRP.yaml').resolve()
graph = Pipeline.from_uri(f'{pipeline_file}#stage2-recalibrate').to_graph()
node = graph.tasks['fgcmBuildFromIsolatedStar']
task = node.task_class(config=node.config)
print('pipeline:', pipeline_file, '| referenceCCD:', task.config.referenceCCD, '| primaryBands:', list(task.config.primaryBands))
real_rows = []
first_traceback = None
for use_science in (False, True):
    for ref in refs:
        visit = int(ref.dataId['visit'])
        err = ''
        try:
            task.fgcmMakeVisitCatalog(camera, {visit: [butler.getDeferred(ref), None]}, useScienceDetectors=use_science)
        except Exception as exc:
            err = f'{type(exc).__name__}: {str(exc)[:150]}'
            if first_traceback is None:
                first_traceback = traceback.format_exc()
        real_rows.append(dict(visit=visit, use_science=use_science, error=err))
real = pd.DataFrame(real_rows)
real.to_csv(DATA_DIR / 'real_fgcm_visit_catalog.csv', index=False)
print(real.groupby('use_science')['error'].apply(lambda s: int((s != '').sum())).rename('n_failed_visits').to_string())
print(real[real.error != ''].groupby(['use_science', 'error']).size().to_string())
if first_traceback:
    print(first_traceback)

numpy 2.4.6 | fgcmcal imported from /pbs/home/d/dagoret/rubin_user/REPO/fgcmcal/python/lsst/fgcmcal/__init__.py
detectors in the camera: 205 | nCcd in the look-up table: 189
pipeline: /cvmfs/sw.lsst.eu/almalinux-x86_64/lsst_distrib/w_2026_46/conda/envs/lsst-scipipe-14.0.0-exact/share/eups/Linux64/drp_pipe/g7715199d20+993d0b9996/pipelines/LSSTCam/DRP.yaml | referenceCCD: 40 | primaryBands: ['i', 'r', 'g', 'z', 'y', 'u']


/cvmfs/sw.lsst.eu/almalinux-x86_64/lsst_distrib/w_2026_46/conda/envs/lsst-scipipe-14.0.0-exact/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:3824: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
/cvmfs/sw.lsst.eu/almalinux-x86_64/lsst_distrib/w_2026_46/conda/envs/lsst-scipipe-14.0.0-exact/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)
/cvmfs/sw.lsst.eu/almalinux-x86_64/lsst_distrib/w_2026_46/conda/envs/lsst-scipipe-14.0.0-exact/lib/python3.13/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
